# PyTorch backend walkthrough

`pyvinecopulib.torch` is a soft-imported subpackage that mirrors the C++ `Bicop` / `Vinecop` evaluation chain in pure PyTorch. The same TLL pair-copula and R-vine math, but everything lives in `torch.nn.Module`s with all the niceties that come with it:

- moveable to GPU with `.to(device)`,
- composable with autograd-aware downstream code,
- one set of weights for the whole vine, easy to plug into a larger neural pipeline.

This notebook walks through the public API end-to-end.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import pyvinecopulib as pv
from pyvinecopulib.torch import TorchBicop, TorchVinecop

torch.set_num_threads(1)
rng = np.random.default_rng(42)

## 1. Bicop — fit and evaluate a TLL pair-copula in PyTorch

We fit a TLL pair-copula on Gaussian-copula samples. There are two equivalent paths:

- `TorchBicop.from_bicop(cop)` — lift a fitted `pv.Bicop` into a `nn.Module`. The fit lives on the C++ side.
- `TorchBicop.from_data(u)` — fit entirely in PyTorch (the same TLL constant-method KDE as C++, ported to torch). Matches the C++ fit to ~1e-11 on the values grid.

In [ ]:
n = 2000
cop_true = pv.Bicop(family=pv.families.gaussian, parameters=np.array([[0.6]]))
u_fit_np = cop_true.simulate(n, seeds=[1, 2, 3])
u_fit_t = torch.from_numpy(u_fit_np)

# Path 1: lift a fitted pv.Bicop
cop_cpp = pv.Bicop.from_data(
  u_fit_np,
  controls=pv.FitControlsBicop(family_set=[pv.families.tll], num_threads=1),
)
bc_via_cpp = TorchBicop.from_bicop(cop_cpp)

# Path 2: pure-torch fit
bc_torch = TorchBicop.from_data(u_fit_t)

# The two fits agree on the underlying values grid up to ~1e-11.
abs_diff = (
  (bc_via_cpp.interp_grid.values - bc_torch.interp_grid.values)
  .abs()
  .max()
  .item()
)
print(f"max |values_cpp - values_torch| = {abs_diff:.2e}")

### Two grid types

`grid_type='normal'` (the default, matches C++) stores the density on a Φ-spaced grid — ~equispaced in z-space, clustered near 0 and 1 in u-space.

`grid_type='linear'` uses `linspace(0, 1, m)` for storage — uniform in u-space. The KDE still evaluates on the same z-range so bandwidth selection is unaffected, only the storage grid changes. The win: O(1) cell-finding (`floor(u * (m-1))`) instead of `searchsorted`. The cost: slightly less resolution near the boundaries.

In [ ]:
bc_normal = TorchBicop.from_data(u_fit_t, grid_type="normal")
bc_linear = TorchBicop.from_data(u_fit_t, grid_type="linear")

u1, u2 = torch.meshgrid(
  torch.linspace(0.02, 0.98, 80, dtype=torch.float64),
  torch.linspace(0.02, 0.98, 80, dtype=torch.float64),
  indexing="ij",
)
uu = torch.stack([u1.reshape(-1), u2.reshape(-1)], dim=-1)
pdf_normal = bc_normal.pdf(uu).reshape(80, 80).numpy()
pdf_linear = bc_linear.pdf(uu).reshape(80, 80).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
levels = np.linspace(0, max(pdf_normal.max(), pdf_linear.max()), 12)
axes[0].contourf(u1, u2, pdf_normal, levels=levels)
axes[0].set_title("grid_type='normal'")
axes[1].contourf(u1, u2, pdf_linear, levels=levels)
axes[1].set_title("grid_type='linear'")
axes[2].contourf(u1, u2, pdf_normal - pdf_linear, cmap="RdBu")
axes[2].set_title("normal - linear")
for ax in axes:
  ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### `cache_integrals=True` — one bilinear interp per eval

With `cache_integrals=False` (the default), `cdf` and `hfunc` are computed on the fly via trapezoidal integration, and `hinv` runs a fixed-iter ITP root-finder. With `cache_integrals=True`, all five of `{cdf, hfunc1, hfunc2, hinv1, hinv2}` are precomputed on the (m, m) grid at construction time; each eval is then one bilinear interp lookup. The trade-off is ~1e-3 mean / ~1e-2 max precision vs the on-the-fly path (the bilinear-interp gap).

In [ ]:
import time

bc_no_cache = TorchBicop.from_data(u_fit_t, cache_integrals=False)
bc_cached = TorchBicop.from_data(u_fit_t, cache_integrals=True)
u_eval = torch.from_numpy(rng.uniform(0.05, 0.95, size=(50_000, 2)))


def time_call(fn, repeats=10):
  fn()  # warmup
  t0 = time.perf_counter()
  for _ in range(repeats):
    fn()
  return (time.perf_counter() - t0) / repeats * 1000


for op in ("hfunc1", "hinv1"):
  t_no = time_call(lambda op=op: getattr(bc_no_cache, op)(u_eval))
  t_ca = time_call(lambda op=op: getattr(bc_cached, op)(u_eval))
  print(f"{op:>8}: cache=False {t_no:7.2f} ms, cache=True {t_ca:7.2f} ms")

## 2. Vinecop — pure-torch fit on a known structure

`TorchVinecop.from_data(u, structure)` fits all pair-copulas of the vine in a tree-by-tree cascade, with each pair fitted via `TorchBicop.from_data`. The structure is taken as input (typically from `pv.Vinecop.from_data` for structure selection, or built explicitly).

The fitted `TorchVinecop` exposes `pdf`, `rosenblatt`, and `inverse_rosenblatt`, all in PyTorch.

In [ ]:
d = 8
n_vine = 2000
# Build a parametric Gaussian R-vine to simulate from.
structure = pv.RVineStructure.simulate(d, seeds=[7])
pair_copulas = [
  [
    pv.Bicop(
      family=pv.families.gaussian,
      parameters=np.array([[0.6 * 0.7**t]]),
    )
    for _ in range(d - t - 1)
  ]
  for t in range(d - 1)
]
cop_true = pv.Vinecop.from_structure(
  structure=structure, pair_copulas=pair_copulas
)
u_vine = cop_true.simulate(n_vine, seeds=[10, 11, 12])

# Fit a TorchVinecop on the simulated data, passing the known structure.
vc = TorchVinecop.from_data(
  torch.from_numpy(u_vine), structure, cache_integrals=True
)
print("trunc_lvl =", vc.trunc_lvl, "  d =", vc.d)

### Round-trip: `inverse_rosenblatt(rosenblatt(u)) ≈ u`

Standard sanity check. The Rosenblatt transform sends dependent uniforms `u` to independent uniforms; its inverse should round-trip up to ITP precision (cache_integrals=True relaxes this slightly because the inverse is interpolated).

In [ ]:
u_round = torch.from_numpy(rng.uniform(0.05, 0.95, size=(500, d)))
w = vc.rosenblatt(u_round)
back = vc.inverse_rosenblatt(w)
max_err = (back - u_round).abs().max().item()
print(f"max round-trip error = {max_err:.2e}")

### Eval strategies: `impl=` and `batched=`

Every public method (`pdf`, `rosenblatt`, `inverse_rosenblatt`) accepts two orthogonal kwargs:

- **`impl='legacy'`** (default): direct port of the C++ cascade with dense (n, d) scratch matrices. Byte-equal to `pv.Vinecop` byte-for-byte on the same fit.
- **`impl='lazy'`**: dict-based pseudo-obs materialization with reference-counted GC. Same math, smaller peak memory.
- **`batched=True`**: stacks all pair-copulas at a tree level and fires one batched bicop call per level instead of N loop iterations. Works for `pdf` and `rosenblatt`. Does NOT work for `inverse_rosenblatt` (the inverse cascade has 2-D deps that don't reduce to per-tree-level waves) — calling it raises `NotImplementedError`.

In [ ]:
u_eval_v = torch.from_numpy(rng.uniform(0.05, 0.95, size=(10_000, d)))
for impl in ("legacy", "lazy"):
  for batched in (False, True):
    t = time_call(lambda i=impl, b=batched: vc.pdf(u_eval_v, impl=i, batched=b))
    print(f"pdf  impl={impl:<7} batched={batched!s:<5}: {t:6.2f} ms")
# inverse_rosenblatt batched=True raises:
try:
  vc.inverse_rosenblatt(u_eval_v, batched=True)
except NotImplementedError as e:
  print("inverse_rosenblatt(batched=True) raises:", str(e).split(".")[0])

## 3. GPU + autograd

Because everything is a `nn.Module`, the usual PyTorch idioms work: move to GPU with `.to(device)`, compose with downstream layers, run backward. Here's a tiny example: a torch vine feeds its pdf into a downstream affine layer, and a gradient flows back through the vine evaluation chain into the layer's parameters.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
vc_dev = vc.to(device)
u_dev = u_eval_v[:512].to(device)

scale = torch.nn.Parameter(
  torch.tensor(1.0, device=device, dtype=torch.float64)
)
bias = torch.nn.Parameter(torch.tensor(0.0, device=device, dtype=torch.float64))

with torch.enable_grad():
  log_pdf = vc_dev.pdf(u_dev).clamp_min(1e-20).log()
  obj = (scale * log_pdf + bias).mean()
  obj.backward()

print("d obj / d scale =", scale.grad.item())
print("d obj / d bias  =", bias.grad.item())
# move back to CPU for the rest of the notebook
_ = vc.to("cpu")

## Caveats

Known limits of `pyvinecopulib.torch` as of this version:

- **Continuous variables only.** No discrete-margin support yet.
- **TLL family only** for the kernel pair-copulas; parametric families (Gaussian, Clayton, …) aren't ported. `TorchVinecop.from_data` errors on a vine with non-TLL / non-indep pair-copulas.
- **No rotations.** TLL pair-copulas always have `rotation=0` in vinecopulib; non-zero rotations would need a per-pair input rotation step that isn't implemented.
- **Single batch.** Each `pdf` / `rosenblatt` / `inverse_rosenblatt` call processes one `(n, d)` tensor; no leading batch dim.
- **`inverse_rosenblatt(…, batched=True)` raises.** The inverse cascade's 2-D dependency graph doesn't reduce to per-tree-level wavefronts. Use `batched=False` (the default).

Most of these are planned to relax in future versions. See `tests/test_torch_*.py` and `scripts/bench_torch_*.py` for the test and benchmark surfaces.